In [61]:
import io
import re
import pandas as pd
from pathlib import Path
pd.set_option("display.max_columns", None)     
pd.set_option("display.max_rows", None)        
pd.set_option("display.max_colwidth", None)  
pd.set_option("display.width", None)       

In [62]:
FOLDER = Path.cwd() / "facebook data"
OUTPUT = Path.cwd()

In [63]:
def read_lines(name):
    """Facebook exports are UTF-16 text files."""
    with open(FOLDER / name, "rb") as f:
        return f.read().decode("utf-16").splitlines()

def to_df(lines):
    return pd.read_csv(io.StringIO("\n".join(lines)))

In [64]:
POST_COLUMNS = {
    "Post ID": "post_id",
    "Publish time": "publish_time",
    "Post type": "post_type",
    "Title": "title",
    "Permalink": "permalink",
    "Views": "views",
    "Viewers": "viewers",
    "Views from Organic posts": "views_organic",
    "Views from Boosted posts": "views_boosted",
    "Reactions": "reactions",
    "Comments": "comments",
    "Shares": "shares",
    "Total clicks": "total_clicks",
    "Link Clicks": "link_clicks",
    "Other Clicks": "other_clicks",
    "Negative feedback from users": "negative_feedback",
}


# A blank in these means "none happened", so they are filled with 0
COUNT_COLUMNS = ["views_organic", "views_boosted", "reactions", "comments", "shares",
                 "total_clicks", "link_clicks", "other_clicks", "negative_feedback"]

DAILY_FILES = {
    "Facebook_Views.csv": "views",
    "Facebook_Viewers.csv": "viewers",
    "Facebook_Visits.csv": "visits",
    "Facebook_Follows.csv": "follows",
    "Facebook_Interactions.csv": "interactions",
    "Facebook_Link clicks.csv": "link_clicks",
}


1.Post performance: one row per post

In [65]:
def build_posts():
    raw = pd.read_csv(FOLDER / "Facebook_Content.csv",
                      encoding="utf-8-sig", dtype={"Post ID": str})
    posts = raw[list(POST_COLUMNS)].rename(columns=POST_COLUMNS)

    posts["title"] = posts["title"].fillna("").str.replace(r"\s+", " ", regex=True).str.strip()
    posts[COUNT_COLUMNS] = posts[COUNT_COLUMNS].fillna(0).astype(int)

    time = pd.to_datetime(posts["publish_time"], format="%m/%d/%Y %H:%M")
    posts["publish_time"] = time.dt.strftime("%H:%M")
    posts["publish_date"] = time.dt.strftime("%Y-%m-%d")
    posts["publish_year"] = time.dt.year
    posts["publish_month"] = time.dt.month
    posts["publish_month_name"] = time.dt.month_name()
    posts["publish_day"] = time.dt.day
    posts["publish_hour"] = time.dt.hour
    posts["weekday"] = time.dt.day_name()

    posts["engagement"] = posts[["reactions", "comments", "shares"]].sum(axis=1)
    posts["engagement_rate_pct"] = (posts["engagement"] / posts["views"] * 100).round(2)
    posts["title_length"] = posts["title"].str.len()

    q1, q3 = posts["views"].quantile([0.25, 0.75])
    posts["is_high_outlier"] = posts["views"] > q3 + 1.5 * (q3 - q1)

    return posts.sort_values("publish_time").reset_index(drop=True)

2. Daily performance: one row per day

In [66]:
def build_daily(posts):
    frames = []
    for file, metric in DAILY_FILES.items():
        df = to_df(read_lines(file)[2:])                       # skip 'sep=,' and title
        frames.append(df.rename(columns={"Date": "date", "Primary": metric})
                        .set_index("date")[metric])

    daily = pd.concat(frames, axis=1)
    daily.index = pd.to_datetime(daily.index)
    daily = daily.sort_index().rename_axis("date")

    daily["year"] = daily.index.year
    daily["month"] = daily.index.month
    daily["month_name"] = daily.index.month_name()
    daily["day"] = daily.index.day
    daily["weekday"] = daily.index.day_name()
    daily["views_7day_avg"] = daily["views"].rolling(7, min_periods=1).mean().round(1)
    daily["cumulative_follows"] = daily["follows"].cumsum()

    posts_per_day = posts.groupby(pd.to_datetime(posts["publish_date"])).size()
    daily["posts_published"] = posts_per_day.reindex(daily.index, fill_value=0)

    daily.index = daily.index.strftime("%Y-%m-%d")
    return daily.reset_index()

3. Audience profile: who follows the page

In [67]:
def build_audience():
    lines = read_lines("Facebook_Audience.csv")

    # Age & gender table
    age = to_df(lines[2:lines.index("")])
    age = age.rename(columns={age.columns[0]: "age"})
    age = age.melt(id_vars="age", var_name="gender", value_name="pct")
    age["gender"] = age["gender"].str.lower()
    age["dimension"] = "age_gender"
    age["category"] = age["gender"] + " " + age["age"]

    def top_list(title, dimension):
        """Block layout: a title line, a row of names, then a row of values."""
        i = lines.index(f'"{title}"')
        block = pd.read_csv(io.StringIO("\n".join(lines[i + 1:i + 3])), header=None).T
        return pd.DataFrame({"dimension": dimension,
                             "category": block[0],
                             "pct": block[1].astype(float)})

    audience = pd.concat([age, top_list("Top cities", "city"),
                          top_list("Top countries", "country")], ignore_index=True)
    audience["pct"] = audience["pct"].round(2)
    return audience[["dimension", "category", "gender", "age", "pct"]]


Build and save

In [68]:
posts = build_posts()
daily = build_daily(posts)
audience = build_audience()

for name, table in {"post_performance": posts,
                    "daily_performance": daily,
                    "audience_profile": audience}.items():
    table.to_csv(OUTPUT / f"{name}.csv", index=False, encoding="utf-8-sig")
    print(f"{name}.csv: {table.shape}")

post_performance.csv: (152, 27)
daily_performance.csv: (315, 15)
audience_profile.csv: (28, 5)


In [69]:
posts.head()

,post_id,publish_time,post_type,title,permalink,views,viewers,views_organic,views_boosted,reactions,comments,shares,total_clicks,link_clicks,other_clicks,negative_feedback,publish_date,publish_year,publish_month,publish_month_name,publish_day,publish_hour,weekday,engagement,engagement_rate_pct,title_length,is_high_outlier
0,1559941502797520,03:52,Links,Stay tuned for information for our holiday open house!,https://www.facebook.com/permalink.php?story_fbid=pfbid023XfSaNQvszApK6NNRW775CqAUQWPCMGNo3qEWhgYh7VohwTbyodLDMqbVBGnuFedl&id=100063450566143,194,107,194,0,0,0,0,12,12,0,0,2026-10-27,2026,10,October,27,3,Tuesday,0,0.00,54,False
1,1401254345332904,04:00,Links,"🌸✨ So many exciting things happening this May at Infinite Blooms! ✨🌸 We have a full lineup of fun events and beautiful blooms just in time for Mother’s Day 💐 🌷 May 5 – Mother’s Day Workshop Come create something special for Mom (or with Mom!) 🛍️ May 7 – Sip & Shop Grab a drink and enjoy a relaxing evening of shopping local 🍧 May 8 – Collaboration with Sassy Slushy’s Sweet treats + fresh flowers… the perfect combo! Order early to reserve a delivery spot! 🌼 Starting Monday May 4th We’ll have a gorgeous selection of hanging baskets and potted flowers available—perfect for gifting or sprucing up your porch just in time for Mother’s Day! Stop in, shop, and celebrate with us—we can’t wait to see you! 💛 📍 Infinite Blooms 🌐 infinitebloomsflorals.com 📞 765-301-5011",https://www.facebook.com/permalink.php?story_fbid=pfbid02u5kFHDQtcznbp5y4PxMwSE4KU17a4XqYu8JYyTwNdCs7BjV325B5sBrxjqwSHbfXl&id=100063450566143,1122,630,1122,0,19,0,7,17,14,3,0,2026-04-30,2026,4,April,30,4,Thursday,26,2.32,766,False
2,1534469415344729,04:06,Photos,"These bright orange Gerbera daisies are just begging to be made into a beautiful bouquet! 🧡🌼 Send one to someone who needs a pick-me-up, surprise a friend, brighten someone’s desk… or order one just because you deserve fresh flowers too! 💐 Custom bouquets available 🚗 Delivery available 🌸 Made fresh at Infinite Blooms No special occasion needed—flowers ARE the occasion. 🧡 Message us or order online and we’ll create something beautiful for you! Infinite Blooms — Every Petal Tells A Story. 765-301-5011",https://www.facebook.com/permalink.php?story_fbid=pfbid02f9bF6EjYr9XKHG1fsn7N8fYjoJG2yqRzQN5g6ZL9XUxiJAsVj5rMik1rbV3okpJLl&id=100063450566143,637,397,637,0,12,1,2,10,0,9,0,2026-10-01,2026,10,October,1,4,Thursday,15,2.35,504,False
3,1414118734046465,04:13,Photos,"We loved creating this custom balloon and floral display for a special 70th birthday celebration at DePauw! 🖤🤍✨ From the black, white, and gold balloons Marley Williams to the fresh florals, this was fun to create. Thank you for letting Infinite Blooms be a part of your special event. Planning a birthday, graduation, shower, or special event? Message us to create something one of a kind! #InfiniteBlooms #DePauw #BirthdayCelebration #BalloonArch #EventFlorals #GreencastleIN #CustomEvents",https://www.facebook.com/permalink.php?story_fbid=pfbid02c59D2K8ujHmB5m8PfEud5a4dmLRmreFdkLnPgx8tgnJWbEQ6a73LD9CfpDzudRfTl&id=100063450566143,2592,1550,2592,0,60,10,4,101,0,73,0,2026-05-15,2026,5,May,15,4,Friday,74,2.85,491,False
4,1526594289465575,04:40,Photos,"A Beautiful Fall Wedding! 💐🧡💜 We had the pleasure of creating the florals for a gorgeous outdoor wedding overlooking the golf course! Vibrant purples, pops of coral and orange, and lush greenery made the perfect combination for this beautiful fall celebration. The colors were absolutely stunning against the outdoor backdrop! Thank you for letting Infinite Blooms be part of your special day! We love bringing each couple’s unique vision to life. 💕 🌸 Every Petal Tells a Story. #InfiniteBlooms #FallWedding #OutdoorWedding #WeddingFlorals #GolfCourseWedding #IndianaFlorist #GreencastleIndiana",https://www.facebook.com/permalink.php?story_fbid=pfbid0vfbMgt3WpAo6kqhPLZrCStHszCPThe3wpVD4G3hDM49ui6aDee1Dm1uKTcBSWXeql&id=100063450566143,1423,829,1423,0,24,2,3,157,0,48,0,20